In [1]:
import requests
import os
import pandas as pd
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer


In [2]:
def process_financial_data(data):
    # Extract the time series data
    time_series_data = data.get('Time Series (5min)', {})
    
    processed_data = {time: {
        "open": float(details["1. open"]),
        "high": float(details["2. high"]),
        "low": float(details["3. low"]),
        "close": float(details["4. close"]),
        "volume": int(details["5. volume"])
    } for time, details in time_series_data.items()}
    
    return pd.DataFrame.from_dict(processed_data, orient='index')

In [3]:
def process_news_data(data):
    articles = data.get('articles', [])
    return [{'title': article['title'], 'description': article['description']} for article in articles]

In [4]:
def fetch_financial_data(symbol):
    ALPHA_VANTAGE_API_KEY = 'KSR0XHEWLKH5EE5X'
    url = f"https://www.alphavantage.co/query?function=TIME_SERIES_INTRADAY&symbol={symbol}&interval=5min&apikey={ALPHA_VANTAGE_API_KEY}"
    response = requests.get(url)
    data = response.json()

    if "Time Series (5min)" not in data:
        raise ValueError("API call failed or returned unexpected data structure:", data.get("Note", data.get("Information", "")))
    df = process_financial_data(data)
    return df

In [7]:
data= fetch_financial_data('AAPL')
data

,open,high,low,close,volume
2024-05-09 19:55:00,184.750,184.798,184.660,184.660,4684
2024-05-09 19:50:00,184.785,184.860,184.730,184.750,3441
2024-05-09 19:45:00,184.820,184.860,184.750,184.860,7082
2024-05-09 19:40:00,184.785,184.840,184.730,184.800,720
2024-05-09 19:35:00,184.795,184.840,184.700,184.800,2441
...,...,...,...,...,...
2024-05-09 12:00:00,183.550,183.640,183.449,183.570,456262
2024-05-09 11:55:00,183.738,183.737,183.515,183.541,288455
2024-05-09 11:50:00,183.680,183.760,183.660,183.732,267702
2024-05-09 11:45:00,183.765,183.770,183.612,183.685,310188


In [18]:
def fetch_news_data(query):
    NEWS_API_KEY = 'ce339a03ac454c3c82bf3bf656a11079'
    url = f"https://newsapi.org/v2/everything?q={query}&apiKey={NEWS_API_KEY}"
    response = requests.get(url)
    data = response.json()

    news_items = process_news_data(data)
    df = pd.DataFrame(news_items)
    return df

In [19]:
news= fetch_news_data('Apple')

In [26]:
def analyze_sentiments(news_df):
    analyzer = SentimentIntensityAnalyzer()
    news_df['title'] = news_df['title'].astype(str)
    news_df['description'] = news_df['description'].astype(str)
    news_df['text'] = news_df['title'] + " " + news_df['description']
    news_df['sentiment'] = news_df['text'].apply(lambda x: analyzer.polarity_scores(x)['compound'])

    return news_df



In [29]:
# Assuming `news_df` is your DataFrame loaded from a file or constructed from data
df2 =analyze_sentiments(news)
df2


,title,description,text,sentiment
0,[Removed],[Removed],[Removed] [Removed],0.0000
1,Apple discounts MLS Season Pass to $69 for the...,You can get an MLS Season Pass for Apple TV at...,Apple discounts MLS Season Pass to $69 for the...,0.0772
2,[Removed],[Removed],[Removed] [Removed],0.0000
3,Apple doesn’t understand why you use technology,Apple’s marketing campaign for its new iPad su...,Apple doesn’t understand why you use technolog...,0.2263
4,It doesn’t matter how many Vision Pro headsets...,"Earlier this week, noted Apple analyst Ming-Ch...",It doesn’t matter how many Vision Pro headsets...,0.2500
...,...,...,...,...
95,"Translation Tech Is Amazing, Except When It’s Not",We can hold surprisingly deep conversations ac...,"Translation Tech Is Amazing, Except When It’s ...",0.8225
96,[Removed],[Removed],[Removed] [Removed],0.0000
97,Apple issues mercenary spyware threat alert,Apple has sent out threat notifications to use...,Apple issues mercenary spyware threat alert Ap...,-0.8692
98,Apple Store Employees in New Jersey File to Un...,Apple employees at the Apple Store in Short Hi...,Apple Store Employees in New Jersey File to Un...,-0.1027


In [30]:
def calculate_financial_trend(financial_df):
    """Calculate financial trend based on recent price movements."""
    # Simple moving averages (SMA) for the short and long term
    short_window = 5
    long_window = 20
    financial_df['short_mavg'] = financial_df['close'].rolling(window=short_window, min_periods=1).mean()
    financial_df['long_mavg'] = financial_df['close'].rolling(window=long_window, min_periods=1).mean()

    # Trend determination
    if financial_df['short_mavg'].iloc[-1] > financial_df['long_mavg'].iloc[-1]:
        return "Uptrend", financial_df['short_mavg'].iloc[-1] - financial_df['long_mavg'].iloc[-1]
    else:
        return "Downtrend", financial_df['short_mavg'].iloc[-1] - financial_df['long_mavg'].iloc[-1]


In [31]:
calculate_financial_trend(data)

('Downtrend', -0.13919999999998822)

In [36]:
def calculate_sentiment_strength(news_df):
    """Calculate the average sentiment and its strength from a DataFrame."""
    if 'sentiment' not in news_df.columns:
        raise ValueError("DataFrame must contain 'sentiment' column")

    average_sentiment = news_df['sentiment'].mean()
    sentiment_strength = news_df['sentiment'].std()

    return average_sentiment, sentiment_strength


In [37]:
calculate_sentiment_strength(df2)

(0.12299900000000001, 0.4403818167731775)

In [39]:
def combine_financial_news(financial_df, news_sentiments):
    
    financial_trend, trend_strength = calculate_financial_trend(financial_df)
    average_sentiment, sentiment_strength = calculate_sentiment_strength(news_sentiments)
    overall_sentiment = "Positive" if average_sentiment > 0 else "Negative"

    if financial_trend == "Uptrend" and overall_sentiment == "Positive":
        prediction = "Strong Bullish"
    elif financial_trend == "Downtrend" and overall_sentiment == "Negative":
        prediction = "Strong Bearish"
    else:
        if abs(trend_strength) > abs(average_sentiment):
            prediction = "Mild " + financial_trend
        else:
            prediction = "Mildly " + overall_sentiment

    
    financial_df['market_trend'] = prediction  # Assigning prediction to a new column

    return financial_df 

In [40]:
combine_financial_news(data,df2)

,open,high,low,close,volume,short_mavg,long_mavg,market_trend
2024-05-09 19:55:00,184.750,184.798,184.660,184.660,4684,184.660000,184.660000,Mild Downtrend
2024-05-09 19:50:00,184.785,184.860,184.730,184.750,3441,184.705000,184.705000,Mild Downtrend
2024-05-09 19:45:00,184.820,184.860,184.750,184.860,7082,184.756667,184.756667,Mild Downtrend
2024-05-09 19:40:00,184.785,184.840,184.730,184.800,720,184.767500,184.767500,Mild Downtrend
2024-05-09 19:35:00,184.795,184.840,184.700,184.800,2441,184.774000,184.774000,Mild Downtrend
...,...,...,...,...,...,...,...,...
2024-05-09 12:00:00,183.550,183.640,183.449,183.570,456262,183.637400,183.948650,Mild Downtrend
2024-05-09 11:55:00,183.738,183.737,183.515,183.541,288455,183.602200,183.895700,Mild Downtrend
2024-05-09 11:50:00,183.680,183.760,183.660,183.732,267702,183.609600,183.860800,Mild Downtrend
2024-05-09 11:45:00,183.765,183.770,183.612,183.685,310188,183.617600,183.826050,Mild Downtrend


In [54]:
import joblib
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
def read_and_process_financial_data(filename):
    df = pd.read_parquet(filename)
    return df

def read_and_process_news_data(filename):
    df = pd.read_parquet(filename)
    sentiments = analyze_sentiments(df)
    return sentiments


In [56]:
def prepare_data_ml(stock_file, news_file):
    financial_data = read_and_process_financial_data(stock_file)
    news_sentiments = read_and_process_news_data(news_file)
    
    # Combine financial data and news sentiments
    combined_data = combine_financial_news(financial_data, news_sentiments)
    
    X = combined_data.drop("market_trend", axis=1)
    y = combined_data["market_trend"]
    
    # Split data into train and test sets
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    return X_train, y_train, X_test, y_test

In [57]:
def train_model(X_train, y_train):
    numeric_features = ["open", "high", "low", "close", "volume"]
    numeric_transformer = Pipeline(
        steps=[
            ("Median Imputer", SimpleImputer(strategy="median")),
            ("Standardization", StandardScaler()),
        ]
    )

    preprocessor = ColumnTransformer(
        transformers=[("Numeric Transformer", numeric_transformer, numeric_features)],
        remainder="drop",
    )

    pipeline = Pipeline(
        steps=[
            ("Preprocessor", preprocessor),
            ("Classifier", RandomForestClassifier(n_estimators=100, random_state=42)),
        ]
    )

    param_grid = {
        "Classifier__max_depth": [5, 10, 15],
        "Classifier__min_samples_split": [2, 5],
    }

    grid_search = GridSearchCV(pipeline, param_grid, cv=5, scoring="accuracy")
    grid_search.fit(X_train, y_train)
    return grid_search

In [61]:

companies = {
    "Apple": "AAPL",
    "Microsoft": "MSFT",
    "Nvidia": "NVDA",
    "AMD": "AMD",
    "Tesla": "TSLA"
}

data_dir = "data/"
models_dir = "models/"

for company, symbol in companies.items():
    stock_file = os.path.join(data_dir, f"{symbol}_financial_data.parquet")
    news_file = os.path.join(data_dir, f"{company}_news_data.parquet")

    if not os.path.exists(stock_file) or not os.path.exists(news_file):
        print(f"Data files missing for {company}. Please check your data directory.")
        continue

    print(f"\nTraining model for {company}:")
    
    financial_data = read_and_process_financial_data(stock_file)

    news_sentiments = read_and_process_news_data(news_file)
    combined_data = combine_financial_news(financial_data, news_sentiments)
    print("Columns in combined data:", combined_data.columns.tolist())
    
    X_train, y_train, X_test, y_test = prepare_data_ml(stock_file, news_file)
    # print("Columns in X_train:", X_train.columns.tolist())
    # print(y_train)

    # Train the model using the training data
    model = train_model(X_train, y_train)
    print(f"Model saved for {company}")
    test_accuracy = model.score(X_test, y_test)
    print(f"Test Accuracy for {company}: {test_accuracy:.2f}")
    if hasattr(model, 'best_score_'):
        print(f"CV Accuracy for {company}: {model.best_score_:.2f}")



Training model for Apple:
Columns in combined data: ['open', 'high', 'low', 'close', 'volume', 'short_mavg', 'long_mavg', 'market_trend']
Model saved for Apple
Test Accuracy for Apple: 1.00
CV Accuracy for Apple: 1.00

Training model for Microsoft:
Columns in combined data: ['open', 'high', 'low', 'close', 'volume', 'short_mavg', 'long_mavg', 'market_trend']
Model saved for Microsoft
Test Accuracy for Microsoft: 1.00
CV Accuracy for Microsoft: 1.00

Training model for Nvidia:
Columns in combined data: ['open', 'high', 'low', 'close', 'volume', 'short_mavg', 'long_mavg', 'market_trend']
Model saved for Nvidia
Test Accuracy for Nvidia: 1.00
CV Accuracy for Nvidia: 1.00

Training model for AMD:
Columns in combined data: ['open', 'high', 'low', 'close', 'volume', 'short_mavg', 'long_mavg', 'market_trend']
Model saved for AMD
Test Accuracy for AMD: 1.00
CV Accuracy for AMD: 1.00

Training model for Tesla:
Columns in combined data: ['open', 'high', 'low', 'close', 'volume', 'short_mavg', 'l

In [67]:
for company, symbol in companies.items():
    stock_file = os.path.join(data_dir, f"{symbol}_financial_data.parquet")
    news_file = os.path.join(data_dir, f"{company}_news_data.parquet")
    model_file = os.path.join(models_dir, f"{symbol}_model.joblib")

    print(f"\nEvaluating model for {company}:")
    print("-" * 60)

    if not os.path.exists(stock_file) or not os.path.exists(news_file) or not os.path.exists(model_file):
        print("Missing data/model files for {company}. Please check your directories.")
        continue

    # Read and prepare data
    financial_data = read_and_process_financial_data(stock_file)
    news_sentiments = read_and_process_news_data(news_file)
    combined_data = combine_financial_news(financial_data, news_sentiments)
    combined_df = pd.DataFrame(combined_data)

    # 'market_trend' is the target column
    X_test = combined_df.drop(['market_trend'], axis=1)
    y_test = combined_df['market_trend']

    # Load model and evaluate
    model = joblib.load(model_file)
    try:
        test_score = model.score(X_test, y_test)
        print(f"Test Accuracy: {test_score:.2f}")
        if test_score < 0.95:
            print("!!!!!!!!!!!! WARNING: Model accuracy below 95% !!!!!!!!!!!!")
    except Exception as e:
        print(f"Error evaluating model for {company}: {str(e)}")


Evaluating model for Apple:
------------------------------------------------------------
Test Accuracy: 1.00

Evaluating model for Microsoft:
------------------------------------------------------------
Test Accuracy: 1.00

Evaluating model for Nvidia:
------------------------------------------------------------
Test Accuracy: 1.00

Evaluating model for AMD:
------------------------------------------------------------
Test Accuracy: 1.00

Evaluating model for Tesla:
------------------------------------------------------------
Test Accuracy: 1.00
